# 03 · Data wrangling with dplyr & tidyr

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: R* - Instructor: Anderson Santos

## Setup — run this first

This cell installs the R packages the lessons use and makes the example data available. On **Google Colab** it just works: run it (Shift+Enter) and wait until it prints **Ready**. You do not need to understand it.

In [ ]:
# Setup — run this first. Works on Google Colab and on a local Jupyter with R.
# It installs the packages the lessons use and makes the example data available.

pkgs <- c("readr", "dplyr", "tidyr", "tibble", "ggplot2", "forcats", "stringr", "vegan")
to_install <- setdiff(pkgs, rownames(installed.packages()))
if (length(to_install) > 0) {
  # On Colab (Linux) use Posit Public Package Manager binaries: ~1-2 min instead of
  # compiling from source (~15-20 min).
  if (grepl("linux", R.version$os) && file.exists("/etc/os-release")) {
    cn <- grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE)
    cn <- gsub('VERSION_CODENAME=|"', "", cn)
    if (length(cn) == 1 && nzchar(cn)) {
      options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", cn)))
      options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
              paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
    }
  }
  install.packages(to_install)
}

# Fetch the example data if it is not already next to the notebook (the Colab case).
if (!file.exists("../data/sample_metadata.csv") && !file.exists("data/sample_metadata.csv")) {
  system("git clone -q https://github.com/andersonavilasantos/ufz-summerschool-r.git course_repo")
  setwd("course_repo/notebooks")
}

# Load the packages quietly and keep read_csv output tidy (cleaner notebook output).
suppressPackageStartupMessages(invisible(lapply(pkgs, require, character.only = TRUE)))
options(readr.show_col_types = FALSE)

cat("Ready. Data folder:", if (file.exists("../data/sample_metadata.csv")) "../data" else "data", "\n")

## Learning objectives

This is the **core** of the session — the toolkit you will reuse all week.

- Master the six core **dplyr verbs**: `filter`, `select`, `mutate`, `arrange`,
  `summarise`, and `group_by`.
- Combine tables with **joins** (`left_join`).
- Move between **wide** and **long** shapes with `pivot_longer` / `pivot_wider`.
- Tidy text and factors with **stringr** and **forcats** along the way.
- Put it together to compute a real result: **taxonomic composition per region**.

---

## 0 · Setup

In [ ]:
library(readr)     # read_csv: import the CSVs
library(dplyr)     # the wrangling verbs (filter, select, mutate, group_by, …)
library(tidyr)     # reshape wide <-> long (pivot_longer / pivot_wider)
library(stringr)   # tidy text columns (str_*)
library(forcats)   # reorder/relabel factors (fct_*)

# Pick the data folder that exists, so the lesson runs from notebooks/ or the root.
data_dir <- if (file.exists("../data/sample_metadata.csv")) "../data" else "data"

# Load the three linked tables of our HITChip study.
meta     <- read_csv(file.path(data_dir, "sample_metadata.csv"))  # one row per sample
taxonomy <- read_csv(file.path(data_dir, "taxonomy.csv"))         # genus -> phylum/family
abund    <- read_csv(file.path(data_dir, "genus_abundance.csv"))  # sample x 130 genera

# Re-impose the clinical BMI order once (as in lesson 02), so every group_by respects it.
bmi_levels <- c("underweight","lean","overweight","obese","severeobese","morbidobese")
meta <- meta |> mutate(bmi_group = factor(bmi_group, levels = bmi_levels))

glimpse(meta)      # quick check: columns, types, first values

---

## 1 · `filter()` — keep the rows you want

`filter()` keeps rows that meet a condition. Conditions use `==`, `>`, `<`, `>=`,
`!=`, and combine with `&` (and), `|` (or).

In [ ]:
# Keep only rows where nationality equals "Scandinavia" ('==' tests equality).
meta |> filter(nationality == "Scandinavia")

# Multiple conditions separated by a comma are combined with "and" (both must hold).
meta |> filter(bmi_group == "lean", age > 50)   # lean and older than 50

# Pipe the filtered rows into nrow() to just count them, not print them.
meta |> filter(diversity_shannon > 6) |> nrow() # how many high-diversity samples?

Two very common needs: dropping missing values and matching a **set** of values.

In [ ]:
meta |> filter(!is.na(age)) |> nrow()              # '!is.na(age)' = keep rows where age is known
meta |> filter(nationality %in% c("US", "UKIE"))   # '%in%' = value is one of this set

### Exercise 1

Keep only participants who are **female**, from **CentralEurope**, and whose
`age` is known. How many are there?

<details><summary><b>Solution</b> (click to expand)</summary>


```r
meta |>
  filter(sex == "female", nationality == "CentralEurope", !is.na(age)) |>
  nrow()
```
</details>

---

## 2 · `select()` — keep the columns you want

In [ ]:
meta |> select(sample_id, age, bmi_group)      # keep only these three columns, in this order
meta |> select(-subject_id)                    # a leading '-' drops a column, keep the rest
meta |> select(sample_id, starts_with("div"))  # starts_with(): match columns by name prefix

`rename()` changes a column name (`new = old`):

In [ ]:
meta |> rename(shannon = diversity_shannon) |> names()  # rename(new = old); names() confirms it

---

## 3 · `mutate()` — add or change columns

`mutate()` creates new columns from existing ones. Every row is computed at once.

In [ ]:
meta <- meta |>
  mutate(
    # cut() bins a numeric column into categories; 'breaks' are the cut points,
    # 'labels' name the 4 resulting age bands. Turns continuous age -> a factor.
    age_band = cut(age,
                   breaks = c(0, 30, 45, 60, 100),
                   labels = c("<30", "30-44", "45-59", "60+")),
    # A comparison makes a TRUE/FALSE column flagging high-diversity samples.
    high_diversity = diversity_shannon > 5.8
  )

# Show just the relevant columns to see the two new ones next to their source.
meta |> select(sample_id, age, age_band, diversity_shannon, high_diversity) |> head()

`stringr` helps when a column is **text**. Here we make a tidy, lower-case region
label and a short code:

In [ ]:
meta <- meta |>
  mutate(
    # str_replace_all() swaps text using a named vector of "old" = "new" rules.
    region_label = str_replace_all(nationality, c("UKIE" = "UK & Ireland",
                                                   "US"   = "United States")),
    # str_sub(x, 1, 3) takes characters 1-3; str_to_upper() upper-cases them.
    region_code  = str_sub(nationality, 1, 3) |> str_to_upper()  # e.g. "Scandinavia" -> "SCA"
  )

# distinct() drops duplicate rows so we see one example of each region label.
meta |> select(nationality, region_label, region_code) |> distinct() |> head()

### Exercise 2

Add a column `diversity_z` giving each sample's Shannon diversity as a
**z-score** (subtract the mean, divide by the standard deviation). Then show the
5 samples with the highest `diversity_z`.
*Hint:* `mean(x)`, `sd(x)`, and `arrange(desc(...))` (next section).

<details><summary><b>Solution</b> (click to expand)</summary>


```r
meta |>
  mutate(diversity_z = (diversity_shannon - mean(diversity_shannon)) /
                        sd(diversity_shannon)) |>
  arrange(desc(diversity_z)) |>
  select(sample_id, diversity_shannon, diversity_z) |>
  head(5)
```
</details>

---

## 4 · `arrange()` — sort the rows

In [ ]:
meta |> arrange(age) |> head(3)                      # sort ascending -> youngest first
meta |> arrange(desc(diversity_shannon)) |> head(3)  # desc() = descending -> most diverse first
meta |> arrange(nationality, desc(age)) |> head(3)   # sort by region, then oldest within region

---

## 5 · `group_by()` + `summarise()` — the core of analysis

This pattern answers almost every "…per group?" question. `group_by()` splits the
data into groups; `summarise()` collapses each group to one row of statistics.

In [ ]:
# group_by() tags the rows by nationality; summarise() then collapses each group to one row.
# (group_by must come first — summarise operates within whatever groups are set.)
meta |>
  group_by(nationality) |>
  summarise(
    n            = n(),                     # n() counts the rows (samples) in each group
    mean_shannon = mean(diversity_shannon), # average diversity for that region
    mean_age     = mean(age, na.rm = TRUE)  # average age; na.rm skips missing ages
  ) |>
  arrange(desc(mean_shannon))               # sort regions from most to least diverse

Group by **two** variables for a cross-tabulated summary:

In [ ]:
meta |>
  filter(!is.na(bmi_group)) |>                  # ignore samples with unknown BMI
  group_by(nationality, bmi_group) |>           # one group per region x BMI combination
  summarise(n = n(), .groups = "drop") |>       # count each combo; .groups="drop" ungroups after
  head(8)

> **Instructor note.** `summarise()` is where the biology appears: "mean
> diversity per region", "richness per BMI class". Emphasise the mental model of
> split, apply, combine.

### Exercise 3

For each `bmi_group` (excluding `NA`), report the number of samples and the
**median** Shannon diversity, sorted from most to least diverse.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
meta |>
  filter(!is.na(bmi_group)) |>
  group_by(bmi_group) |>
  summarise(n = n(), median_shannon = median(diversity_shannon)) |>
  arrange(desc(median_shannon))
```
</details>

---

## 6 · Reshaping: `pivot_longer()` and `pivot_wider()`

Our abundance table is **wide** (130 genus columns). Most tidyverse tools — and
all of ggplot2 — prefer **long** (tidy) data: one row per *observation*. Here an
observation is "one genus in one sample".

In [ ]:
# pivot_longer stacks many columns into two: one for the old names, one for the values.
abund_long <- abund |>
  pivot_longer(
    cols      = -sample_id,   # pivot every column except sample_id (the '-' excludes it)
    names_to  = "genus",      # the old column names (genus names) fill this new column
    values_to = "abundance"   # the old cell values (counts) fill this new column
  )

abund_long          # tall table: 1006 samples x 130 genera = 130,780 rows, one per pair

Now every genus/sample pair is a row — perfect for joining, grouping and
plotting. To go back the other way, `pivot_wider()` spreads a long table wide:

In [ ]:
abund_long |>
  filter(genus %in% c("Akkermansia", "Escherichia coli et rel.")) |>  # just two genera
  # pivot_wider is the inverse: values from 'genus' become column names again.
  pivot_wider(names_from = genus, values_from = abundance) |>
  head()

> **Instructor note.** "Long for computing and plotting, wide for reading and
> matrices." This single idea covers most day-to-day omics wrangling.

---

## 7 · Joins — connecting tables

`abund_long` knows the *genus* but not its *phylum*. The `taxonomy` table has
that mapping. `left_join()` glues columns from `taxonomy` onto matching rows of
`abund_long`, matching on the shared `genus` column.

In [ ]:
# left_join keeps every row of the left table (abund_long) and pastes on matching
# taxonomy columns where the 'genus' key agrees.
abund_tax <- abund_long |>
  left_join(taxonomy, by = "genus")     # each row now also knows its phylum & family

abund_tax |> select(sample_id, genus, phylum, abundance) |> head()

Now join the **sample metadata** too, matching on `sample_id`, so every row knows
its participant's region and BMI group. This one tidy table combines all three
files:

In [ ]:
# Join the metadata too, matching on sample_id. We select() first so we only pull in
# the few metadata columns we need (not all 7) — keeps the joined table lean.
full <- abund_tax |>
  left_join(meta |> select(sample_id, nationality, bmi_group, age),
            by = "sample_id")

glimpse(full)      # every row = one genus, in one sample, with its taxonomy + participant info

> **Instructor note.** `left_join` keeps every row of the left table and adds
> matching info from the right. If a genus were missing from `taxonomy`, its
> `phylum` would be `NA`, which is a quick way to catch mismatches.

### Exercise 4

Using `abund_tax`, compute the **total abundance per phylum** across the whole
study, sorted from most to least abundant. Which phylum dominates the human gut
here?
*Hint:* `group_by(phylum)` then `summarise(total = sum(abundance))`.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
abund_tax |>
  group_by(phylum) |>
  summarise(total = sum(abundance)) |>
  arrange(desc(total))
# Firmicutes and Bacteroidetes dominate — the classic gut signature.
```
</details>

---

## 8 · Putting it together — composition per region

Now we chain everything into one analysis: **the mean relative abundance of each
phylum, per geographic region.** This is the kind of multi-step pipe you will
write daily.

In [ ]:
composition <- full |>
  # 1. collapse the many genera into their phylum: total abundance per sample x phylum
  group_by(sample_id, nationality, phylum) |>
  summarise(phy_abund = sum(abundance), .groups = "drop") |>
  # 2. re-group by sample and divide by the sample's own total -> a proportion (0-1)
  group_by(sample_id) |>
  mutate(rel_abund = phy_abund / sum(phy_abund)) |>  # relative abundance within the sample
  ungroup() |>                                       # clear grouping before the next step
  # 3. now average those per-sample proportions across all samples of a region
  group_by(nationality, phylum) |>
  summarise(mean_rel = mean(rel_abund), .groups = "drop")

# Reshape to a human-readable table: one row per phylum, one column per region, in %.
composition |>
  mutate(mean_pct = round(mean_rel * 100, 1)) |>          # fraction -> rounded percentage
  select(phylum, nationality, mean_pct) |>
  pivot_wider(names_from = nationality, values_from = mean_pct)  # regions become columns

Read across a row to see how a phylum's share shifts between regions — a genuine
ecological pattern, produced with the six verbs plus a join and two pivots. We
**visualise** exactly this table in lesson 04.

In [ ]:
# Save for reuse / inspection.
saveRDS(composition, file.path(tempdir(), "composition.rds"))

### Exercise 5 (challenge)

Reorder the `phylum` factor by its **overall** mean relative abundance (most
abundant first) using `forcats::fct_reorder()`, then show the composition table
again. *Hint:* first compute an overall mean per phylum, or use `fct_reorder`
directly on the long `composition` with `mean_rel`.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
composition |>
  mutate(phylum = fct_reorder(phylum, mean_rel, .fun = sum, .desc = TRUE)) |>
  arrange(phylum, nationality) |>
  head(10)
```
</details>

---

## Recap

- **`filter`** rows, **`select`** columns, **`mutate`** new columns,
  **`arrange`** to sort.
- **`group_by` + `summarise`** = split–apply–combine: the answer to every
  "per-group" question.
- **`pivot_longer` / `pivot_wider`** move between tidy-long and readable-wide.
- **`left_join`** connects tables on a shared key (`genus`, `sample_id`).
- **stringr** cleans text, **forcats** orders factors.
- Chaining these with the pipe turns three raw files into a real result:
  **phylum composition per region**.

**Next:** lesson **04** — turn these tables into figures with **ggplot2**.